# Pawnee National Grassland Land Swap
## Pawnee Grassland Boundaries

- **Objective:**  
For this notebook, different shapefiles are queried to create different boundaries for federal and state parcels as well as a master and western Pawnee boundary shapefile to be used in all proceeding notebooks.  

- **Objective goals:**
    - Compile and process boundary datasets for the Pawnee National Grassland study area  
    - Create a unified **master boundary** for spatial analysis  
    - Generate subset boundaries (e.g., western Pawnee) for targeted analyses  
    - Prepare clean, reusable boundary layers for downstream workflows  

- **Author:** Max Warnock  
- **Code review and/or edits:** Kayleigh Ward  
- **Date:** April 9, 2026  
- **Last date of revision:** April 28, 2026  

---

### 🛠️ Prerequisites & Setup

**Mandatory Libraries:**
- `geopandas`
- `pandas`
- `shapely`
- `matplotlib`
- `requests` (for API data access)

**Environment:**
- Conda environment (e.g., `earth-analytics-python`) with geospatial dependencies installed  
- Internet connection required for querying parcel data  

**Data Sources:**
- USFS Administrative Boundaries (Pawnee National Grassland)  
- Weld County Parcels (ArcGIS REST Feature Service)  
- Supporting boundary shapefiles stored in `/data/boundaries/`  

**Related Notebooks:**
- Downstream notebooks (e.g., land values, species occurrences) depend on outputs from this notebook  

**Notes:**
- All layers are projected to a common CRS prior to processing  
- The master boundary serves as the **spatial constraint** for all analyses in this project  
- Boundary processing is designed to preserve spatial detail while enabling efficient clipping and overlay operations  

---

### 🏗️ Methodology

#### 1. Load and Inspect Boundary Data
- Import USFS boundary shapefiles and parcel data  
- Inspect geometry, CRS, and attribute structure  

#### 2. Query and Prepare Parcel Data
- Retrieve parcel data from Weld County API  
- Filter parcels by ownership (e.g., federal, state)  
- Clean and standardize attribute fields  

#### 3. Reproject and Align Spatial Data
- Convert all datasets to a common CRS  
- Ensure consistent spatial alignment across layers  

#### 4. Create Ownership-Based Boundaries
- Subset parcels by ownership type (federal, state)  
- Dissolve geometries where appropriate to create unified ownership boundaries  

#### 5. Construct Master Boundary
- Combine relevant boundary layers  
- Dissolve into a single **master boundary** polygon  
- Validate geometry and remove artifacts  

#### 6. Generate Subset Boundaries
- Clip master boundary to create the **western Pawnee boundary**  
- Export all finalized boundary layers for reuse  

---

### Reproducibility Notes
- All file paths are relative to the project root directory  
- Outputs are written to `/data/boundaries/boundary-data-final/`  
- This notebook should be run prior to any analysis notebooks that require spatial constraints  

---

### ⚡ Troubleshooting/Notes
- Ensure CRS consistency before performing spatial operations (common source of empty outputs) for this notebook  
- Large parcel datasets may take time to load and process  
- API queries may fail intermittently—rerun cells if needed  
- If geometries appear distorted, verify projection and transformation steps  

# Libraries

In [1]:
### file paths, OS operations, utilities
import os
import pathlib
import zipfile
import time
from glob import glob
from getpass import getpass

### data handling 
import pandas as pd
import geopandas as gpd

### web requests / data download
import requests

### geospatial visualization 
import holoviews as hv
import hvplot.pandas
import cartopy.crs as ccrs

### GBIF API access
import pygbif.occurrences as occ
import pygbif.species as species

# Primary Directory

In [2]:
### set up root file path
# Walk up from the current directory to find the repo root (contains .git)
_cwd = pathlib.Path(os.getcwd()).resolve()
repo_root = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / '.git').exists()),
    _cwd
)
os.chdir(repo_root)

data_dir = os.path.join(repo_root, 'data')
os.makedirs(data_dir, exist_ok=True)

print(f'Repo root: {repo_root}')

Repo root: C:\Users\warno\Documents\GitHub\Pawnee-Grasslands-Project


# Secondary Directories

In [3]:
### set a directory for the National Grassland boundary data
boundary_dir = os.path.join(data_dir, 'boundaries')
os.makedirs(boundary_dir, exist_ok=True)


### set an inner directory for boundary data processing
bound_process_dir = os.path.join(boundary_dir, 'boundary-data-processing')
os.makedirs(bound_process_dir, exist_ok=True)


### set a directory for the master boundary data
master_boundary_dir = os.path.join(bound_process_dir, 'master_boundary')
os.makedirs(master_boundary_dir, exist_ok=True)


### set a directory for the federal boundary data
federal_boundary_dir = os.path.join(bound_process_dir, 'federal_boundary')
os.makedirs(federal_boundary_dir, exist_ok=True)


### set a directory for the state boundary data
state_boundary_dir = os.path.join(bound_process_dir, 'state_boundary')
os.makedirs(state_boundary_dir, exist_ok=True)

### set a directory for the county parcel data
county_parcel_dir = os.path.join(bound_process_dir, 'county_parcels')
os.makedirs(county_parcel_dir, exist_ok=True)

### Master boundary data download

In [4]:
### Google Drive url
master_boundary_url = f"https://drive.google.com/uc?export=download&id=1gsL1tzZu6ZH28b3PQm4VgZzlY2GrkO8p"

### zip path
download_path = os.path.join(master_boundary_dir, "pawnee_master_boundary.zip")

### create a session
session = requests.Session()

### request the file
response = session.get(master_boundary_url, stream=True)
response.raise_for_status()

### save the zip file
with open(download_path, "wb") as f:
    for chunk in response.iter_content(chunk_size=8192):
        if chunk:
            f.write(chunk)

print(f"Downloaded to: {download_path}")

### unzip the file
with zipfile.ZipFile(download_path, "r") as zip_ref:
    zip_ref.extractall(master_boundary_dir)

print(f"Extracted to: {master_boundary_dir}")

Downloaded to: C:\Users\warno\Documents\GitHub\Pawnee-Grasslands-Project\data\boundaries\boundary-data-processing\master_boundary\pawnee_master_boundary.zip
Extracted to: C:\Users\warno\Documents\GitHub\Pawnee-Grasslands-Project\data\boundaries\boundary-data-processing\master_boundary


In [5]:
### path to extracted folder
master_boundary_dir = os.path.join(master_boundary_dir, "pawnee_master_boundary")

### build full path
pawnee_master_boundary_path = os.path.join(master_boundary_dir, "pawnee_master_boundary.shp")

### read the shapefile
pawnee_master_boundary_gdf = gpd.read_file(pawnee_master_boundary_path)

### check it
pawnee_master_boundary_gdf.head()

,Id,Location,geometry
0,0,1.0,"POLYGON ((-11593123.97 5012571.209, -11542474...."
1,0,NaN,"POLYGON ((-11614609.509 4955044.708, -11616750..."


In [6]:
### make sure its projected in EPSG 4326
pawnee_master_boundary_gdf = pawnee_master_boundary_gdf.to_crs(epsg=4326)

### plot with hvplot
pawnee_master_boundary_gdf.hvplot(
    geo=True,
    tiles="EsriImagery",
    title="Pawnee National Grassland Boundaries",
    line_color="white",
    line_width=2,
    fill_alpha=0,
    width=600,
    height=500
)

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]

### Download Weld County Parcel data

In [12]:
### url
county_parcel_url = "https://services.arcgis.com/ewjSqmSyHJnkfBLL/arcgis/rest/services/Parcels_open_data/FeatureServer/replicafilescache/Parcels_open_data_542388106199933781.zip"

### local zip path
zip_path = os.path.join(county_parcel_dir, "county_parcels.zip")

### download
response = requests.get(county_parcel_url, stream=True)
response.raise_for_status()

with open(zip_path, "wb") as f:
    for chunk in response.iter_content(8192):
        if chunk:
            f.write(chunk)

print("Downloaded:", zip_path)

Downloaded: C:\Users\warno\Documents\GitHub\Pawnee-Grasslands-Project\data\boundaries\boundary-data-processing\county_parcels\county_parcels.zip


In [13]:
### unzip
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(county_parcel_dir)

print("Extracted to:", county_parcel_dir)

Extracted to: C:\Users\warno\Documents\GitHub\Pawnee-Grasslands-Project\data\boundaries\boundary-data-processing\county_parcels


In [14]:
### path to extracted file
county_parcel_path = os.path.join(county_parcel_dir, "Parcels_open_data.shp")

### read the file
county_parcel_gdf = gpd.read_file(county_parcel_path)

### check it
county_parcel_gdf.head()

,PARCEL,MHSPACE,ACCOUNTTYP,ACCOUNTNO,NAME,ADDRESS1,ADDRESS2,CITY,STATE,ZIPCODE,...,Shape_Leng,RECEPTION_,AddressPre,LGLANDASD,LGIMPASD,TOTALLGASD,SCLANDASD,SCIMPASD,TOTALSCASD,geometry
0,002919000001,NaN,R,R0000186,GREEN ELSIE KLINGINSMITH (1/3 INT),530 MCKINLEY ST,NaN,STERLING,CO,807512637,...,18720.439972,NaN,None,1790.0,0.0,1790.0,1790.0,0.0,1790.0,"POLYGON ((-11541970.912 5012592.769, -11540907..."
1,002919000002,NaN,R,R0000286,U S A,2850 YOUNGFIELD ST,NaN,LAKEWOOD,CO,802157210,...,1320.086368,NaN,None,10030.0,0.0,10030.0,10030.0,0.0,10030.0,"POLYGON ((-11542213.325 5011302.221, -11542213..."
2,002920000003,NaN,R,R0000386,GREEN ELSIE KLINGINSMITH (1/3 INT),530 MCKINLEY ST,NaN,STERLING,CO,807512637,...,17354.419416,NaN,None,1820.0,0.0,1820.0,1820.0,0.0,1820.0,"POLYGON ((-11538243.903 5012270.018, -11538249..."
3,002921000001,NaN,R,R0000586,GREEN ELSIE KLINGINSMITH (1/3 INT),530 MCKINLEY ST,NaN,STERLING,CO,807512637,...,17300.695352,NaN,None,1980.0,0.0,1980.0,1980.0,0.0,1980.0,"POLYGON ((-11536130.965 5012279.379, -11536129..."
4,002922000002,NaN,R,R0000686,SHEFFLER RANCH LLC,65295 COUNTY ROAD 135,NaN,NEW RAYMER,CO,807429645,...,17239.487033,NaN,None,1790.0,8090.0,9880.0,1790.0,8910.0,10700.0,"POLYGON ((-11534006.724 5012304.442, -11534006..."


### Select federal parcels

In [ ]:
### select only federal owned parcels
pawnee_fed = county_parcel_gdf[county_parcel_gdf["NAME"] == "U S A"].copy()

### check the result
pawnee_fed

In [29]:
### make sure its projected in EPSG 4326
pawnee_fed = pawnee_fed.to_crs(epsg=4326)

### clip to the master boundary
pawnee_fed = gpd.clip(
    pawnee_fed,
    pawnee_master_boundary_gdf
)

### plot with hvplot
pawnee_fed.hvplot(
    geo = True,
    tiles = 'EsriImagery',
    title = 'Pawnee National Grassland Federally Owned Boundaries',
    fill_color = None,
    line_color = "white",
    frame_width = 600
)

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]

### Select state owned parcels

In [30]:
### select only state owned parcels
pawnee_state = county_parcel_gdf[county_parcel_gdf["NAME"] == "COLORADO STATE OF"].copy()

### check the result
pawnee_state

,PARCEL,MHSPACE,ACCOUNTTYP,ACCOUNTNO,NAME,ADDRESS1,ADDRESS2,CITY,STATE,ZIPCODE,...,Shape_Leng,RECEPTION_,AddressPre,LGLANDASD,LGIMPASD,TOTALLGASD,SCLANDASD,SCIMPASD,TOTALSCASD,geometry
16429,072701000012,NaN,R,R1102786,COLORADO STATE OF,1127 N SHERMAN ST STE 300,NaN,DENVER,CO,802032398,...,16867.607235,NaN,None,56490.0,0.0,56490.0,56490.0,0.0,56490.0,"MULTIPOLYGON (((-103.60064 40.61046, -103.5994..."
5430,053332000019,NaN,R,R0491686,COLORADO STATE OF,1127 N SHERMAN ST STE 300,NaN,DENVER,CO,802032398,...,1005.535921,NaN,None,5110.0,0.0,5110.0,5110.0,0.0,5110.0,"POLYGON ((-103.67593 40.61014, -103.67702 40.6..."
5566,053536000004,NaN,R,R0507986,COLORADO STATE OF,1127 N SHERMAN ST STE 300,NaN,DENVER,CO,802032398,...,21110.448333,NaN,None,81120.0,0.0,81120.0,81120.0,0.0,81120.0,"POLYGON ((-103.69627 40.61717, -103.69621 40.6..."
5433,053333000004,NaN,R,R0491986,COLORADO STATE OF,1127 N SHERMAN ST STE 300,NaN,DENVER,CO,802032398,...,13223.919627,NaN,None,2520.0,0.0,2520.0,2520.0,0.0,2520.0,"POLYGON ((-103.64828 40.61733, -103.64832 40.6..."
5443,053336000007,NaN,R,R0492986,COLORADO STATE OF,1127 N SHERMAN ST STE 300,NaN,DENVER,CO,802032398,...,15902.906898,NaN,None,40950.0,0.0,40950.0,40950.0,0.0,40950.0,"POLYGON ((-103.58149 40.61864, -103.58155 40.6..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,003536000005,NaN,R,R0018186,COLORADO STATE OF,1127 N SHERMAN ST STE 300,NaN,DENVER,CO,802032398,...,20812.895098,NaN,None,81640.0,0.0,81640.0,81640.0,0.0,81640.0,"POLYGON ((-103.91606 40.96857, -103.91597 40.9..."
60,003136000004,NaN,R,R0009086,COLORADO STATE OF,1127 N SHERMAN ST STE 300,NaN,DENVER,CO,802032398,...,21197.217066,NaN,None,2650.0,0.0,2650.0,2650.0,0.0,2650.0,"POLYGON ((-103.68763 40.97021, -103.68755 40.9..."
22,002934000004,NaN,R,R0004386,COLORADO STATE OF,1127 N SHERMAN ST STE 300,NaN,DENVER,CO,802032398,...,15821.058075,NaN,None,41310.0,0.0,41310.0,41310.0,0.0,41310.0,"POLYGON ((-103.61184 40.97076, -103.61185 40.9..."
24,002936000004,NaN,R,R0004586,COLORADO STATE OF,1127 N SHERMAN ST STE 300,NaN,DENVER,CO,802032398,...,21096.936781,NaN,None,76640.0,0.0,76640.0,76640.0,0.0,76640.0,"POLYGON ((-103.57391 40.96363, -103.57865 40.9..."


In [31]:
### make sure its projected in EPSG 4326
pawnee_state = pawnee_state.to_crs(epsg=4326)

### clip to the master boundary
pawnee_state = gpd.clip(
    pawnee_state,
    pawnee_master_boundary_gdf
)

### plot with hvplot
pawnee_state.hvplot(
    geo = True,
    tiles = 'EsriImagery',
    title = 'Pawnee National Grassland State Owned Boundaries',
    fill_color = None,
    line_color = "white",
    frame_width = 600
)

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]

In [32]:
### make sure its projected in EPSG 4326
county_parcel_gdf = county_parcel_gdf.to_crs(epsg=4326)

### clip to the master boundary
county_parcel_gdf = gpd.clip(
    county_parcel_gdf,
    pawnee_master_boundary_gdf
)

### plot all parcels
county_parcel_gdf.hvplot(
    geo = True,
    tiles = 'EsriImagery',
    title = 'Pawnee National Grassland Parcel Boundaries',
    fill_color = None,
    line_color = "white",
    frame_width = 600
)

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]

In [33]:
### state plot
state_plot = pawnee_state.hvplot(
    geo=True,
    crs=ccrs.PlateCarree(),
    color="blue",
    fill_alpha=0.4,
    line_color="white",
    line_width=1,
    label="State"
)

### federal plot
fed_plot = pawnee_fed.hvplot(
    geo=True,
    crs=ccrs.PlateCarree(),
    color="green",
    fill_alpha=0.4,
    line_color="white",
    line_width=1,
    label="Federal"
)

### master boundary plot
master_plot = pawnee_master_boundary_gdf.hvplot(
    geo=True,
    crs=ccrs.PlateCarree(),
    fill_alpha=0,
    line_color="white",
    line_width=2,
    label="Master Boundary"
)

### create the plot
pawnee_boundary_plot = (hv.element.tiles.EsriImagery() * state_plot * fed_plot * master_plot
).opts(
    title="Pawnee State, Federal, and Master Boundaries",
    width=700,
    height=550,
    legend_position="top_left"
)

### save an interactive html version of the plot

figures_dir = os.path.join(repo_root, 'figures')

boundary_fig_dir = os.path.join(figures_dir, 'boundary_figures')
os.makedirs(boundary_fig_dir, exist_ok=True)

boundary_plot_path = os.path.join(boundary_fig_dir, 'pawnee_boundary_plot.html')
hv.save(pawnee_boundary_plot, boundary_plot_path)

print(f'Saved clipped GBIF map to: {boundary_plot_path}')

pawnee_boundary_plot

Saved clipped GBIF map to: C:\Users\warno\Documents\GitHub\Pawnee-Grasslands-Project\figures\boundary_figures\pawnee_boundary_plot.html


:Overlay
   .Tiles.I                  :Tiles   [x,y]
   .Polygons.State           :Polygons   [Longitude,Latitude]
   .Polygons.Federal         :Polygons   [Longitude,Latitude]
   .Polygons.Master_Boundary :Polygons   [Longitude,Latitude]

### Create file structure for Pawnee processed data

In [27]:
### set a directory for final boundary data
bound_final_dir = os.path.join(boundary_dir, 'boundary-data-final')
os.makedirs(bound_final_dir, exist_ok=True)


### set a directory for the master boundary data
master_bound_final_dir = os.path.join(bound_final_dir, 'master_boundary')
os.makedirs(master_bound_final_dir, exist_ok=True)


### set a directory for the federal boundary data
fed_bound_final_dir = os.path.join(bound_final_dir, 'federal_boundary')
os.makedirs(fed_bound_final_dir, exist_ok=True)


### set a directory for the state boundary data
state_bound_final_dir = os.path.join(bound_final_dir, 'state_boundary')
os.makedirs(state_bound_final_dir, exist_ok=True)


### set a directory for the parcel boundary data
parcel_bound_final_dir = os.path.join(bound_final_dir, 'parcel_boundary')
os.makedirs(parcel_bound_final_dir, exist_ok=True)

### Save processed data in these folders

In [35]:
### save master boundary
pawnee_master_path = os.path.join(master_bound_final_dir, "pawnee_master.shp")
pawnee_master_boundary_gdf.to_file(pawnee_master_path)

### save fed boundary
pawnee_fed_path = os.path.join(fed_bound_final_dir, "pawnee_fed.shp")
pawnee_fed.to_file(pawnee_fed_path)

### save state boundary
pawnee_state_path = os.path.join(state_bound_final_dir, "pawnee_state.shp")
pawnee_state.to_file(pawnee_state_path)

### save state boundary
pawnee_parcel_path = os.path.join(parcel_bound_final_dir, "pawnee_parcel.shp")
county_parcel_gdf.to_file(pawnee_parcel_path)

INFO:Created 2 records
c:\Users\warno\miniconda3\envs\earth-analytics-python\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Field SALEDT created as String field, though DateTime requested.
  ogr_write(
INFO:Created 598 records
c:\Users\warno\miniconda3\envs\earth-analytics-python\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Field SALEDT created as String field, though DateTime requested.
  ogr_write(
INFO:Created 119 records
c:\Users\warno\miniconda3\envs\earth-analytics-python\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Field SALEDT created as String field, though DateTime requested.
  ogr_write(
INFO:Created 3,473 records


### Universal Calls for Pawnee Boundary Data

In [36]:
### root dir
data_dir = os.path.join(repo_root, 'data')

### main boundary dir
boundary_dir = os.path.join(data_dir, 'boundaries')

### boundary dir for processed data for Pawnee
boundary_dir_final = os.path.join(boundary_dir, 'boundary-data-final')


### MASTER
### master boundary dir
master_bound = os.path.join(boundary_dir_final, 'master_boundary')

### master boundary shapefile
master_bound_path = os.path.join(master_bound, "pawnee_master.shp")

### master boundary convert to gdf
master_bound_gdf = gpd.read_file(master_bound_path)


### FEDERAL
### federal boundary dir
federal_bound = os.path.join(boundary_dir_final, 'federal_boundary')

### federal boundary shapefile 
federal_bound_path = os.path.join(federal_bound, 'pawnee_fed.shp')

### federal boundary convert to gdf
federal_bound_gdf = gpd.read_file(federal_bound_path)


### STATE
### state boundary dir
state_bound = os.path.join(boundary_dir_final, 'state_boundary')

### state boundary shapefile 
state_bound_path = os.path.join(state_bound, 'pawnee_state.shp')

### state boundary convert to gdf
state_bound_gdf = gpd.read_file(state_bound_path)


### PARCEL
### parcel boundary dir
parcel_bound = os.path.join(boundary_dir_final, 'parcel_boundary')

### parcel boundary shapefile 
parcel_bound_path = os.path.join(parcel_bound, 'pawnee_parcel.shp')

### parcel boundary convert to gdf
parcel_bound_gdf = gpd.read_file(parcel_bound_path)

### Select only the Western Pawnee area

In [19]:
### select only western pawnee area
pawnee_west = pawnee_master_boundary_gdf[pawnee_master_boundary_gdf["Location"].isna()].copy()
pawnee_west

,Id,Location,geometry
1,0,NaN,"POLYGON ((-104.33581 40.6104, -104.35504 40.61..."


In [20]:
### clip state to the western pawnee
pawnee_state_west = gpd.clip(
    pawnee_state,
    pawnee_west
)

In [21]:
### clip fed to the western pawnee
pawnee_fed_west = gpd.clip(
    pawnee_fed,
    pawnee_west
)

In [22]:
### clip parcels to the western pawnee
pawnee_parcel_west = gpd.clip(
    county_parcel_gdf,
    pawnee_west
)

In [23]:
### plot all three again for the western pawnee

### state plot
state_plot_west = pawnee_state_west.hvplot(
    geo=True,
    crs=ccrs.PlateCarree(),
    color="blue",
    fill_alpha=0.4,
    line_color="white",
    line_width=1,
    label="State"
)

### federal plot
fed_plot_west = pawnee_fed_west.hvplot(
    geo=True,
    crs=ccrs.PlateCarree(),
    color="green",
    fill_alpha=0.4,
    line_color="white",
    line_width=1,
    label="Federal"
)

### master boundary plot west
master_plot_west = pawnee_west.hvplot(
    geo=True,
    crs=ccrs.PlateCarree(),
    fill_alpha=0,
    line_color="white",
    line_width=2,
    label="Master Boundary"
)

parcel_plot_west = pawnee_parcel_west.hvplot(
    geo=True,
    crs=ccrs.PlateCarree(),
    fill_alpha=0,
    line_color='white',
    line_width=1,
    label="Parcels"
)

### create the plot
west_pawnee_plot = (hv.element.tiles.EsriImagery() * parcel_plot_west * state_plot_west * fed_plot_west * master_plot_west
).opts(
    title="West Pawnee State, Federal, and Master Boundaries",
    width=700,
    height=550,
    legend_position="top_left"
)

### save an interactive html version of the plot

west_boundary_plot_path = os.path.join(boundary_fig_dir, 'west_pawnee_boundary_plot.html')
hv.save(west_pawnee_plot, west_boundary_plot_path)

print(f'Saved clipped GBIF map to: {west_boundary_plot_path}')

west_pawnee_plot

Saved clipped GBIF map to: C:\Users\warno\Documents\GitHub\Pawnee-Grasslands-Project\figures\boundary_figures\west_pawnee_boundary_plot.html


:Overlay
   .Tiles.I                  :Tiles   [x,y]
   .Polygons.Parcels         :Polygons   [Longitude,Latitude]
   .Polygons.State           :Polygons   [Longitude,Latitude]
   .Polygons.Federal         :Polygons   [Longitude,Latitude]
   .Polygons.Master_Boundary :Polygons   [Longitude,Latitude]

### Create file structure for Pawnee West processed data

In [ ]:
### set a directory for final boundary west data 
bound_final_dir_west = os.path.join(boundary_dir, 'boundary-data-final-west')
os.makedirs(bound_final_dir_west, exist_ok=True)


### set a directory for the master boundary west data
master_bound_final_dir_west = os.path.join(bound_final_dir_west, 'master_boundary_west')
os.makedirs(master_bound_final_dir_west, exist_ok=True)


### set a directory for the federal boundary data
fed_bound_final_dir_west = os.path.join(bound_final_dir_west, 'federal_boundary_west')
os.makedirs(fed_bound_final_dir_west, exist_ok=True)


### set a directory for the state boundary data
state_bound_final_dir_west = os.path.join(bound_final_dir_west, 'state_boundary_west')
os.makedirs(state_bound_final_dir_west, exist_ok=True)


### set a directory for the parcel boundary data
parcel_bound_final_dir_west = os.path.join(bound_final_dir_west, 'parcel_boundary_west')
os.makedirs(parcel_bound_final_dir_west, exist_ok=True)

### Save Pawnee West processed data in these folders

In [25]:
### save master boundary
pawnee_west_path = os.path.join(master_bound_final_dir, "pawnee_master_west.shp")
pawnee_west.to_file(pawnee_west_path)

### save fed boundary
pawnee_fed_west_path = os.path.join(fed_bound_final_dir, "pawnee_fed_west.shp")
pawnee_fed_west.to_file(pawnee_fed_west_path)

### save state boundary
pawnee_state_west_path = os.path.join(state_bound_final_dir, "pawnee_state_west.shp")
pawnee_state_west.to_file(pawnee_state_west_path)

### save state boundary
pawnee_parcel_west_path = os.path.join(parcel_bound_final_dir, "pawnee_parcel_west.shp")
pawnee_parcel_west.to_file(pawnee_parcel_west_path)

INFO:Created 1 records
INFO:Created 1 records
c:\Users\warno\miniconda3\envs\earth-analytics-python\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Field SALEDT created as String field, though DateTime requested.
  ogr_write(
INFO:Created 53 records
c:\Users\warno\miniconda3\envs\earth-analytics-python\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Field SALEDT created as String field, though DateTime requested.
  ogr_write(
INFO:Created 1,189 records


### Universal Calls for Pawnee West Boundary Data

In [26]:
### root dir
data_dir = os.path.join(repo_root, 'data')

### main boundary dir
boundary_dir = os.path.join(data_dir, 'boundaries')

### boundary dir for processed data for Western Pawnee
boundary_dir_west = os.path.join(boundary_dir, 'boundary-data-final-west')


### MASTER
### master boundary dir
master_bound_west = os.path.join(boundary_dir_west, 'master_boundary')

### master boundary shapefile
master_bound_west_path = os.path.join(master_bound_west, "pawnee_master_west.shp")

### master boundary convert to gdf
master_bound_west_gdf = gpd.read_file(master_bound_west_path)


### FEDERAL
### federal boundary dir
federal_bound_west = os.path.join(boundary_dir_west, 'federal_boundary')

### federal boundary shapefile 
federal_bound_west_path = os.path.join(federal_bound_west, 'pawnee_fed_west.shp')

### federal boundary convert to gdf
federal_bound_west_gdf = gpd.read_file(federal_bound_west_path)


### STATE
### state boundary dir
state_bound_west = os.path.join(boundary_dir_west, 'state_boundary')

### state boundary shapefile 
state_bound_west_path = os.path.join(state_bound_west, 'pawnee_state_west.shp')

### state boundary convert to gdf
state_bound_west_gdf = gpd.read_file(state_bound_west_path)


### PARCEL
### parcel boundary dir
parcel_bound_west = os.path.join(boundary_dir_west, 'parcel_boundary')

### parcel boundary shapefile 
parcel_bound_west_path = os.path.join(parcel_bound_west, 'pawnee_parcel_west.shp')

### parcel boundary convert to gdf
parcel_bound_west_gdf = gpd.read_file(parcel_bound_west_path)